# Milestone 12 - Build the agent for real

The **Agents with LangGraph** lessons teach the agent loop, tool calls, memory, human approval
and streaming as plain Python. This notebook builds them with **LangGraph's and LangChain's real
classes**, the ones you will use at work:

| Step | What you use |
|---|---|
| 1. A tool | `@tool` from `langchain_core.tools` |
| 2. The agent loop | `StateGraph`, `MessagesState`, `ToolNode`, `tools_condition` |
| 3. Memory | `InMemorySaver` and a `thread_id` |
| 4. A person approves | `interrupt` and `Command(resume=...)` |
| 5. Streaming | `graph.stream(..., stream_mode="updates")` |
| 6. Tracing | `mlflow.langchain.autolog`, spans of type `TOOL` |

The agent answers questions about a shared Databricks space from its runbooks, and can restart a
cluster, but only after a person says yes.

**How to run it:** `uv run jupyter lab notebooks/m12_build_the_agent_for_real.ipynb`. The tests
use a scripted model, so they give the same result everywhere; set `LIVE = True` to use a model
on your Ollama that can call tools (like `qwen3:14b`) through LangChain's `ChatOpenAI`.

In [ ]:
import os
import tempfile
from pathlib import Path

import ipytest
import mlflow
import pytest
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt
from mlflow.entities import SpanType
from pydantic import ConfigDict

ipytest.autoconfig(clean="test_*")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

RUNBOOKS = {
    "quota": "Out of quota: ask the platform team for a higher limit in the shared space.",
    "restart": "To restart a cluster, its owner restarts it from the Compute page.",
    "access": "For access to a catalog, ask its owner to grant it in Unity Catalog.",
}


class ScriptedChatModel(BaseChatModel):
    """A real LangChain chat model whose replies are written in advance.

    A chat model is a class with a _generate method: messages in, a message out. This one
    returns the next message from its script, so the tests behave the same everywhere. A reply
    with tool_calls asks the graph to run a tool; a reply with text is the answer."""

    model_config = ConfigDict(arbitrary_types_allowed=True)
    script: list
    seen: list = []   # how many messages the model saw on each call

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def bind_tools(self, tools, **kwargs):   # what lets the model call tools
        return self.bind(tools=[convert_to_openai_tool(t) for t in tools], **kwargs)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        self.seen.append(len(messages))
        i = len(self.seen) - 1
        reply = self.script[i] if i < len(self.script) else AIMessage(content="Done.")
        return ChatResult(generations=[ChatGeneration(message=reply)])


def call(name, args, i=1):
    """A model reply that asks for one tool call."""
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": f"call_{i}"}])


LIVE = False
if LIVE:
    from langchain_openai import ChatOpenAI

    live_model = ChatOpenAI(base_url="http://localhost:11434/v1", api_key="ollama", model="qwen3:14b", temperature=0)

## 1. A tool

A **tool** is a function the model may ask to run. `@tool` turns a function into one: its
name becomes the tool's name, its **docstring** becomes the description the model reads to
decide when to use it, and its type hints say what arguments it takes. So write the docstring
for the model, not for yourself.

Write `search_runbooks(query: str) -> str`, decorated with `@tool`, with a one-line docstring.
Return every runbook line whose key appears in the query (lowercased), joined with new lines,
or `"Nothing found."`.

In [ ]:
# TODO: decorate it with @tool, and give it a docstring the model can read.
def search_runbooks(query: str) -> str:
    raise NotImplementedError

In [ ]:
%%ipytest

def test_the_tool_has_a_name_a_description_and_works():
    assert search_runbooks.name == "search_runbooks"
    assert search_runbooks.description, "the docstring is what the model reads"
    assert search_runbooks.invoke({"query": "We are out of QUOTA"}) == RUNBOOKS["quota"]
    assert search_runbooks.invoke({"query": "lunch"}) == "Nothing found."

## 2. The agent loop as a graph

The loop from the lesson: the model replies; if it asked for a tool, run it and give the model
the result; repeat until it answers. In LangGraph:

- `StateGraph(MessagesState)`: a graph whose state is the list of messages so far.
- An `"agent"` node: calls the model (with `model.bind_tools(tools)`) on the messages and
  returns `{"messages": [reply]}`. LangGraph *adds* it to the list.
- A `"tools"` node: `ToolNode(tools)` runs the tool calls in the last message.
- Edges: `START` to `"agent"`; from `"agent"`, `add_conditional_edges("agent", tools_condition)`
  goes to `"tools"` if the reply asked for a tool and to `END` if not; `"tools"` back to `"agent"`.
- `graph.compile(checkpointer=...)` turns it into something you can run.

Write `build_agent(model, tools, checkpointer=None)`.

In [ ]:
def build_agent(model, tools, checkpointer=None):
    # Step 1: bound = model.bind_tools(tools)
    # Step 2: def agent(state: MessagesState): return {"messages": [bound.invoke(state["messages"])]}
    # Step 3: build the graph: two nodes, three edges (one of them conditional), then compile.
    raise NotImplementedError

In [ ]:
%%ipytest

def test_the_agent_calls_the_tool_then_answers():
    model = ScriptedChatModel(script=[call("search_runbooks", {"query": "quota"}), AIMessage(content="Ask the platform team.")])
    graph = build_agent(model, [search_runbooks])
    out = graph.invoke({"messages": [HumanMessage("We are out of quota")]})
    kinds = [type(m).__name__ for m in out["messages"]]
    assert kinds == ["HumanMessage", "AIMessage", "ToolMessage", "AIMessage"]
    assert out["messages"][2].content == RUNBOOKS["quota"]
    assert out["messages"][-1].content == "Ask the platform team."

## 3. Memory

A **checkpointer** saves the graph's state after every step, under a **thread id**. Run the
graph again with the same thread id and it carries on from the saved messages: that is the
agent's memory of the conversation. `InMemorySaver` keeps it in memory; at work you would use a
database-backed one.

Write `ask(graph, thread_id, question)`: invoke the graph with the question as a
`HumanMessage` and the config `{"configurable": {"thread_id": thread_id}}`, and return the last
message's text.

In [ ]:
def ask(graph, thread_id, question):
    raise NotImplementedError

In [ ]:
%%ipytest

def test_the_same_thread_remembers_and_a_new_one_does_not():
    model = ScriptedChatModel(script=[AIMessage(content="Hello."), AIMessage(content="You said hi."), AIMessage(content="New chat.")])
    graph = build_agent(model, [search_runbooks], InMemorySaver())
    assert ask(graph, "t1", "hi") == "Hello."
    assert ask(graph, "t1", "what did I say?") == "You said hi."
    ask(graph, "t2", "hi again")
    assert model.seen == [1, 3, 1], "thread t1 resent the history; thread t2 started fresh"

## 4. A person approves risky actions

Restarting a cluster is not something an agent should do alone. LangGraph can **pause** inside a
node: `interrupt(value)` stops the run, saves the state, and hands `value` to whoever is
watching (here, your test). The run resumes when you invoke the graph again with
`Command(resume=answer)`, and `interrupt(...)` then returns that answer. It needs a
checkpointer, because the paused state has to be saved somewhere.

Write `build_approving_agent(model, tools, checkpointer)`, like `build_agent` but with an
`"approve"` node between the agent and the tools:

- Route `tools_condition`'s `"tools"` outcome to `"approve"`:
  `add_conditional_edges("agent", tools_condition, {"tools": "approve", END: END})`.
- In `approve(state)`: if the last message asks for `"restart_cluster"`, call
  `interrupt({"tool": ..., "args": ...})`. If the answer is no, return
  `Command(goto="agent", update={"messages": [ToolMessage("A person declined this action.", tool_call_id=...)]})`
  (one `ToolMessage` per tool call in the message). Otherwise return `Command(goto="tools")`.

In [ ]:
@tool
def restart_cluster(name: str) -> str:
    """Restart a Databricks cluster by name. A person must approve it first."""
    return f"Restarted {name}."

In [ ]:
def build_approving_agent(model, tools, checkpointer):
    raise NotImplementedError

In [ ]:
%%ipytest

def _run(answer):
    model = ScriptedChatModel(script=[call("restart_cluster", {"name": "etl"}), AIMessage(content="All done.")])
    graph = build_approving_agent(model, [restart_cluster], InMemorySaver())
    config = {"configurable": {"thread_id": "approve"}}
    paused = graph.invoke({"messages": [HumanMessage("restart etl")]}, config)
    assert paused["__interrupt__"][0].value == {"tool": "restart_cluster", "args": {"name": "etl"}}
    return graph.invoke(Command(resume=answer), config)


def test_yes_runs_the_tool():
    out = _run(True)
    assert out["messages"][-2].content == "Restarted etl."


def test_no_never_runs_it():
    out = _run(False)
    assert out["messages"][-2].content == "A person declined this action."
    assert all(m.content != "Restarted etl." for m in out["messages"])

## 5. Streaming

`graph.invoke` waits for the end. `graph.stream(...)` hands you each step as it happens, which
is how a page shows "searching the runbooks..." while the agent works. With
`stream_mode="updates"`, each item is a dict from the node that just ran to what it returned.

Write `stream_steps(graph, question, thread_id)`: return the names of the nodes in the order
they ran.

In [ ]:
def stream_steps(graph, question, thread_id):
    raise NotImplementedError

In [ ]:
%%ipytest

def test_steps_arrive_in_order():
    model = ScriptedChatModel(script=[call("search_runbooks", {"query": "restart"}), AIMessage(content="Its owner restarts it.")])
    graph = build_agent(model, [search_runbooks], InMemorySaver())
    assert stream_steps(graph, "how do I restart a cluster?", "s1") == ["agent", "tools", "agent"]

## 6. Tracing the agent

`mlflow.langchain.autolog()` traces every LangChain and LangGraph run with no other change:
one trace per run, with a span for each model call (type `CHAT_MODEL`) and each tool call
(type `TOOL`). This is how you find out *which* tool an agent called, with what arguments, when
it gets something wrong.

Write `tool_calls_in_last_trace()`: flush the trace queue (`mlflow.flush_trace_async_logging()`),
get the last trace (`mlflow.get_trace(mlflow.get_last_active_trace_id())`), and return the names
of its spans whose `span_type` is `SpanType.TOOL`, in order.

In [ ]:
if os.getenv("DRILL_NOTEBOOK_OFFLINE") == "1":
    folder = Path(tempfile.mkdtemp(prefix="drill-m12-"))
else:
    folder = Path(".drill/m12")
    folder.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{folder / 'mlflow.db'}")
mlflow.set_experiment("m12-build-the-agent")
mlflow.langchain.autolog()

In [ ]:
def tool_calls_in_last_trace():
    raise NotImplementedError

In [ ]:
%%ipytest

def test_the_trace_shows_which_tool_ran():
    model = ScriptedChatModel(script=[call("search_runbooks", {"query": "access"}), AIMessage(content="Ask the owner.")])
    ask(build_agent(model, [search_runbooks], InMemorySaver()), "traced", "how do I get access?")
    assert tool_calls_in_last_trace() == ["search_runbooks"]

## Try it live

With `LIVE = True` in the first cell and a tool-calling model in Ollama, build the approving
agent with `live_model` instead of a scripted one and ask it to restart a cluster:

```python
graph = build_approving_agent(live_model, [search_runbooks, restart_cluster], InMemorySaver())
config = {"configurable": {"thread_id": "live"}}
print(graph.invoke({"messages": [HumanMessage("Please restart the etl cluster")]}, config).get("__interrupt__"))
print(graph.invoke(Command(resume=False), config)["messages"][-1].content)
```

Then open the MLflow UI (`uv run mlflow ui --backend-store-uri sqlite:///.drill/m12/mlflow.db`)
and read the trace: every model call, the tool it asked for, and where it paused.